# 02. Tiền Xử Lý Dữ Liệu

Notebook này thực hiện:
1. Tạo dữ liệu mẫu nhỏ (để test thuật toán)
2. Đọc dataset Yoochoose thật
3. Tiền xử lý: lọc phiên ngắn/dài, item hiếm
4. Chia train/test theo thời gian

In [ ]:
# --- Cho phép import gói src/ (notebook đặt ở thư mục gốc dự án) ---
import sys
from pathlib import Path
_root = Path.cwd()
if not (_root / "src").is_dir() and (_root.parent / "src").is_dir():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
from collections import Counter

from src import config
from src.data import load_clicks, preprocess, save_processed

print("Thư viện + gói src/ đã sẵn sàng!")


## 2.1. Dữ liệu mẫu nhỏ (chạy được ngay, không cần tải gì)

In [ ]:
# Giả lập 5 phiên lịch sử trên website bán sách
sample_sessions = {
    'S1': ['DacNhanTam', 'NghiGiau', 'TuDuyTrieuPhu', '7ThoiQuen'],
    'S2': ['DacNhanTam', 'NghiGiau', 'PythonCoBan'],
    'S3': ['NghiGiau', 'TuDuyTrieuPhu', 'TiengAnhGiaoTiep'],
    'S4': ['PythonCoBan', 'TiengAnhGiaoTiep', 'MachineLearning'],
    'S5': ['DacNhanTam', 'TuDuyTrieuPhu', '7ThoiQuen', 'NgheThuat'],
}

# Phiên hiện tại: khách ẩn danh đang xem 3 cuốn sách
sample_query = ['DacNhanTam', 'NghiGiau', 'TuDuyTrieuPhu']

print('=== DỮ LIỆU MẪU ===')
print(f'Số phiên lịch sử: {len(sample_sessions)}')
print(f'Phiên truy vấn: {sample_query}')
print()
for sid, items in sample_sessions.items():
    print(f'  {sid}: {items}')
print()
print('Câu hỏi: Người dùng đang xem [DacNhanTam, NghiGiau, TuDuyTrieuPhu].')
print('Hệ thống nên gợi ý cuốn sách nào tiếp theo?')

## 2.2. Đọc dataset Yoochoose (dữ liệu thật)

**Hướng dẫn tải:**
1. Vào https://www.kaggle.com/datasets/chadgostopp/recsys-challenge-2015
2. Tải file `yoochoose-clicks.dat`
3. Đặt vào thư mục `data/`

In [ ]:
# Đọc dataset thật qua src.data.load_clicks (tự kiểm tra & in thống kê)
if config.CLICKS_PATH.exists():
    df = load_clicks(config.CLICKS_PATH)
    print(df.head(10))
else:
    print(f"✗ Chưa tìm thấy {config.CLICKS_PATH}")
    print("  → Tải từ: https://www.kaggle.com/datasets/chadgostopp/recsys-challenge-2015")
    print("  → Đặt yoochoose-clicks.dat vào thư mục data/")
    df = None


## 2.3. Lấy mẫu + Tiền xử lý + Chia Train/Test (gọi `src.data.preprocess`)

Hàm `preprocess()` trong `src/data.py` thực hiện đúng quy trình chống rò rỉ dữ liệu (data leakage):
1. **Lấy mẫu 1/64** theo phiên (cố định `seed=42` để tái lập).
2. **Chia theo thời gian:** 90% phiên **sớm nhất** làm train, 10% **muộn nhất** làm test → mô phỏng "dùng quá khứ dự đoán tương lai".
3. **Lọc item hiếm:** chỉ tính tần suất trên **tập train** rồi áp cho cả hai (không nhìn test).
4. **Lọc độ dài** phiên trong khoảng [2, 20].

In [ ]:
if df is not None:
    train_sessions, test_sessions, info = preprocess(
        df,
        sample_fraction=config.SAMPLE_FRACTION,
        min_item_freq=config.MIN_ITEM_FREQ,
        min_len=config.MIN_LEN,
        max_len=config.MAX_LEN,
        train_ratio=config.TRAIN_RATIO,
        seed=config.SEED,
    )
    save_processed(train_sessions, test_sessions, config.PROCESSED_PATH)
    print(f"\nCách chia: {info['split']}")
else:
    print("Chưa có dữ liệu thật → bỏ qua. Xem dữ liệu mẫu ở notebook 03.")
